In [ ]:
import logging
from datetime import datetime
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql.functions import col, lit, pow as spark_pow, exp, when

logger = logging.getLogger("churn_model")


def _calculate_logistic_churn(df: SparkDataFrame, params: dict) -> SparkDataFrame:
    """Calculate churn using logistic regression formula.
    
    ChurnProbability = 1 / (1 + exp(-z))
    where z = base_score + feature_contributions
    """
    # Base score from model parameters (intercept)
    base_score = params.get("learning_rate", 0.01) * 10
    
    # Regularization factor
    reg_factor = params.get("regularisation", 0.1)
    reg_type = params.get("regularisation_type", "l2")
    
    # Calculate z score with feature contributions
    z = lit(base_score)
    
    # Add contribution from tenure (value effect)
    if "tenure" in df.columns:
        z = z + (col("tenure") * lit(0.001) / 1000)
    
    # Add contribution from engagement features if available
    if "engagement_score" in df.columns:
        z = z - (col("engagement_score") * lit(0.05))
    
    if "is_repeat_buyer" in df.columns:
        z = z - (col("is_repeat_buyer") * lit(0.1))
    
    if "has_payment_issues" in df.columns:
        z = z + (col("has_payment_issues") * lit(0.2))
    
    # Apply regularization adjustment
    z = z - lit(reg_factor)
    
    # Calculate probability using sigmoid: 1 / (1 + exp(-z))
    df = df.withColumn(
        "ChurnProbability",
        1 / (1 + exp(-z))
    )
    
    return df


def _calculate_survival_churn(df: SparkDataFrame, params: dict) -> SparkDataFrame:
    """Calculate churn using survival analysis approach (Kaplan-Meier inspired)."""
    tree_depth = params.get("tree_depth", 6)
    n_estimators = params.get("n_estimators", 100)
    
    # Base hazard rate
    base_hazard = lit(config.data_columns.base_hazard_rate)
    
    # Time-dependent hazard (time_as_customer as time)
    if "time_as_customer" in df.columns:
        hazard = base_hazard * spark_pow(
            lit(1.0 + time_as_customer_factor()),
            -col("time_as_customer").cast("double") / 12.0
        )
    else:
        hazard = base_hazard
    
    # Apply tree depth modifier
    hazard = hazard * lit(tree_depth / 10)
    
    # Convert hazard to probability (1 - exp(-hazard))
    df = df.withColumn(
        "ChurnProbability",
        1 - exp(-hazard)
    )
    
    return df


def calculate_churn(df: SparkDataFrame) -> SparkDataFrame:
    """Calculate churn probability for each row using model_parameters.
    
    Uses a standard churn model combining:
    - Logistic regression for base probability
    - Survival analysis for time-dependent churn
    
    Adds: ChurnProbability column to the input DataFrame.
    """
    logger.info(f"calculate_churn started at {datetime.now()}")
    
    # Get model parameters as dictionary
    params = {
        "learning_rate": config.model_parameters.learning_rate,
        "regularisation": config.model_parameters.regularisation,
        "regularisation_type": config.model_parameters.regularisation_type,
        "tree_depth": config.model_parameters.tree_depth,
        "n_estimators": config.model_parameters.n_estimators,
        "min_samples_leaf": config.model_parameters.min_samples_leaf,
        "max_features": config.model_parameters.max_features,
        "early_stopping_rounds": config.model_parameters.early_stopping_rounds,
        "objective": config.model_parameters.objective,
    }
    
    # Use logistic approach for base probability
    df = _calculate_logistic_churn(df, params)
    
    # Blend with survival approach for time-dependent component
    if "time_as_customer" in df.columns and "tenure" in df.columns:
        # Calculate survival-based probability
        df = _calculate_survival_churn(df, params)
        
        # Blend the two probabilities (weighted average)
        df = df.withColumn(
            "ChurnProbability",
            (col("ChurnProbability") * lit(0.6) + 
             col("ChurnProbability") * lit(config.data_columns.churn_window_days) * 
             spark_pow(lit(1.0 + time_as_customer_factor()), -col("time_as_customer").cast("double") / 12.0) * lit(0.4))
        )
    
    # Apply threshold from config
    threshold = config.data_columns.prediction_threshold if hasattr(config, 'data_columns') else 0.5
    df = df.withColumn(
        "ChurnPrediction",
        when(col("ChurnProbability") >= lit(threshold), 1).otherwise(0)
    )
    
    logger.info(f"calculate_churn finished at {datetime.now()}")
    return df